In [4]:
import os
import json
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve, average_precision_score, cohen_kappa_score)
from sklearn.calibration import calibration_curve

In [ ]:
import colab from google.colab import drive
drive.mount('/content/drive')

In [5]:
class Config:
    # Paths, Update as per your local / HPC setup
    data_dir       = r"C:\Users\harimenshi\Documents\python\iris\data"
    chaksu_dir     = r"C:\Users\harimenshi\Documents\python\iris\data\chaksu"
    output_dir     = r"C:\Users\harimenshi\Documents\python\iris\outputs"
    checkpoint_dir = r"C:\Users\harimenshi\Documents\python\iris\outputs\checkpoints"

    # Training - Adjust these parameters as needed
    image_size     = 224
    batch_size     = 32
    num_epochs     = 30
    learning_rate  = 1e-4
    weight_decay   = 5e-4
    patience       = 10

    # Normalization
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    # Device subgroups for fairness evaluation, e.g., different camera devices
    chaksu_devices = ['Bosch', 'Forus', 'Remidio']


config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(config.output_dir, exist_ok=True)

Device: cpu


In [6]:
import os
print(os.listdir(config.data_dir))
print(os.listdir(config.chaksu_dir))

['acrima', 'chaksu', 'origa', 'refuge2020']
['test', 'train', 'val']


In [ ]:
class ApplyCLAHE(object):
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        # Convert PIL Image to numpy array
        img_np = np.array(img)

        # Apply CLAHE to the green channel, as it represents the most clinically significant wavelength for lesion visualization.
        green_channel = img_np[:, :, 1]
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        green_clahe = clahe.apply(green_channel)
        img_np[:, :, 1] = green_clahe

        # Convert back to PIL Image
        img_clahe_pil = Image.fromarray(img_clahe)

        return img_clahe_pil

In [ ]:

if config.compute_normalization: #automatic mean calculation and std calculation
        mean = np.mean([np.array(img).mean(axis=(0, 1)) for img, _ in train_dataset], axis=0) / 255.0
        std = np.std([np.array(img).std(axis=(0, 1)) for img, _ in train_dataset], axis=0) / 255.0
        print(f"Computed mean: {mean}, std: {std}")
else:
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]
        print(f"Using default mean: {mean}, std: {std}")

In [ ]:
def build_transforms(mean, std):
    train_transforms = transforms.Compose([
            ApplyCLAHE(clip_limit=2.0, tile_grid_size=(8, 8)),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    return train_transforms, val_transforms

In [ ]:
train_transforms, test_transforms = build_transforms(mean=config.mean, std=config.std)

In [ ]:
train_dataset = ImageFolder(root=os.path.join(config.chaksu_dir, 'train'), transform=train_transforms)
test_dataset = ImageFolder(root=os.path.join(config.chaksu_dir, 'test'), transform=test_transforms)
val_dataset = ImageFolder(root=os.path.join(config.chaksu_dir, 'val'), transform=test_transforms)  # Assuming you have a validation set

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)  # Using validation set for validation as well

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 256),  # Assuming binary classification
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 2),  # Assuming binary classification

)
model = model.to(device)
print(model.fc) 


In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=config.learning_rate, steps_per_epoch=len(train_loader), epochs=config.num_epochs)
focal_loss = FocalLoss(alpha=0.25, gamma=2.0, reduction='mean')  # Adjust alpha and gamma as needed
scaler = torch.amp.GradScaler('cuda')  # For mixed precision training and efficent memory usage

best_val_accuracy = 0.0
epoch_no_improvement = 0
accuracy = 0.0

start_epoch = 0

if os.path.exists(os.path.join(config.checkpoint_dir, 'checkpoint.pth')):
    checkpoint = torch.load(os.path.join(config.checkpoint_dir, 'checkpoint.pth'), map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_accuracy = checkpoint['best_val_accuracy']
    print(f"Resuming from epoch {start_epoch}")
else:
    print("No checkpoint found, creating a new one.")

for epoch in range(start_epoch, config.num_epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = focal_loss(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        if batch_idx % 10 == 0:  # Print every 10 batches
            print(f"Epoch [{epoch+1}/{config.num_epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
    
    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = focal_loss(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        accuracy = correct / total
        print(f"Validation Loss: {val_loss:.4f}, Accuracy: {accuracy:.4f}")

    if accuracy > best_val_accuracy:
        epoch_no_improvement = 0
        best_val_accuracy = accuracy
        print(f"Saved best model with accuracy: {best_val_accuracy:.4f}")
        torch.save(model.state_dict(), os.path.join(config.checkpoint_dir, 'best_model.pth'))
    else:
        epoch_no_improvement += 1
        if epoch_no_improvement >= config.patience:
            print(f"No improvement for {config.patience} epochs. Stopping early at epoch {epoch+1}.")
            break

    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{config.num_epochs}] completed. Average Loss: {epoch_loss:.4f}")

    torch.save({
    'epoch':                epoch,
    'model_state_dict':     model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_val_accuracy':    best_val_accuracy,
    'val_accuracy':         accuracy,
}, os.path.join(config.checkpoint_dir, 'best_model.pth'))

In [ ]:
checkpoint = torch.load(os.path.join(config.checkpoint_dir, 'best_model.pth'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model - Val Accuracy: {checkpoint['best_val_accuracy']:.4f}")

In [ ]:
def evaluation(model, test_loader, device, dataset_name, class_to_idx, output_dir):
    glaucoma_class_index = class_to_idx.get('glaucoma', None)
    if glaucoma_class_index is None:
        raise ValueError("Glaucoma class index not found in class_to_idx")

    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, output_dict=True)
    tn, fp, fn, tp = cm.ravel()

    results = {
        'confusion_matrix': cm.tolist(),
        'classification_report': report,
        'auc': roc_auc_score(all_labels, all_probs[:, glaucoma_class_index]),
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'kappa': cohen_kappa_score(all_labels, all_preds),
        'labels': all_labels.tolist(),
        'predictions': all_preds.tolist(),
        'probabilities': all_probs.tolist(),
        'dataset': dataset_name
    }
    
    with open(os.path.join(output_dir, f"{dataset_name}_results.json"), 'w') as f:
        json.dump(results, f, indent=4)

    print(f"\nEvaluation Results on {results['dataset']}: {results['dataset']}")
    print(f"Test AUC: {results['auc']:.4f}, Sensitivity: {results['sensitivity']:.4f}, Specificity: {results['specificity']:.4f}, Kappa: {results['kappa']:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Classification Report:\n{json.dumps(report, indent=4)}")

    return results

chaksu_results = evaluation(model, test_loader, device, "Chaksu", train_dataset.class_to_idx, config.output_dir)

In [ ]:
arcmia_dataset = ImageFolder(root=os.path.join(config.data_dir, 'arcmia'), transform=test_transforms)
arcmia_loader = DataLoader(arcmia_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
arcmia_results = evaluation(model, arcmia_loader, device, "ARCMIA", train_dataset.class_to_idx, config.output_dir)

In [ ]:
orgia_dataset = ImageFolder(root=os.path.join(config.data_dir, 'orgia'), transform=test_transforms)
orgia_loader = DataLoader(orgia_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
orgia_results = evaluation(model, orgia_loader, device, "ORGIA", train_dataset.class_to_idx, config.output_dir)

In [ ]:
refuge_dataset = ImageFolder(root=os.path.join(config.data_dir, 'refuge'), transform=test_transforms)
refuge_loader = DataLoader(refuge_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
refuge_results = evaluation(model, refuge_loader, device, "REFUGE", train_dataset.class_to_idx, config.output_dir)